In [1]:
%cd ../../..

/Users/hoangle/Projects/untangling-people/Food-Waste-Optimization


In [2]:
import time
import random
from itertools import combinations, product

import pandas as pd
import polars as pl

from pcs_forecast import forecast_pcs_per_meal

In [3]:
# path = "data/processed/phase_4/menus/vik_2024-12-02.parquet"

# df = pd.read_parquet(path)

# df.head()

In [4]:
seed = time.time()
random.seed(seed)

In [5]:
NUM_VEGAN_PER_DAY = 2
NUM_MEALS_PER_DAY = [3, 4]
NUM_DAY_LEVEL_MENUS = 10_000_000
MAX_MEAL_OCCURENCES = 2
MIN_KELA_PER_DAY = 2
MIN_GLUTEN_FREE_PER_DAY = 1
NUM_DAY_LEVEL_MENUS_2 = 1000
NUM_EACH_COMBO = 1_000_000
NUM_FISH_PER_WEEK = 2

# restaurant = "phy"
restaurant = "che"
# restaurant = "exa"
# restaurant = "vik"
schoolyear = "24-25"
date_start = "2025-02-03"

# Polars

In [6]:
path = "data/processed/phase_4/dim_meals.parquet"
dim_meals_raw = pl.read_parquet(path)
dim_meals_raw.head()

meal_id,meal_type,schoolyear,restaurant,attributes,aliases
i64,str,str,list[str],list[str],list[str]
9017,"""vegan""","""24-25""","[""che"", ""exa"", ""vik""]","[""vegan-miscellaneous"", ""kela""]","[""""Butter"" härkäpapua & pähkinää""]"
7201,"""vegan""","""23-24""",null,[],"[""2023 Härkäpu-sienilasagnette""]"
9032,"""vegan""","""23-24""",null,[],"[""Appelisiini-luomukikhernecurrya""]"
9102,"""vegan""","""23-24""",null,[],"[""Artisokkavugetteja & tuoretomaattisalsaa""]"
7010,"""vegetarian""","""24-25""","[""che"", ""exa"", ""vik""]",[],"[""Aurajuusto-pinaattilasagnette"", ""Aurajuusto-pinaattilasagnettea""]"


In [7]:
path = "data/processed/phase_4/dim_co2.xlsx"
dim_co2 = pl.read_excel(path)
dim_co2.head()

meal_id,co2
i64,f64
34,0.81
37,0.61
710,0.67
713,0.56
724,0.82


In [8]:
path = "data/processed/phase_4/dim_waste.xlsx"
dim_waste = pl.read_excel(path)
dim_waste.head()

meal_id,waste
i64,f64
9017,0.01
7201,0.01
9032,0.01
9102,0.01
7010,0.039256


In [9]:
path = 'data/processed/phase_4/dim_pieces_whole.xlsx'
dim_pcs_whole = pl.read_excel(path)

dim_pcs_whole.head()

date,pcs,restaurant
date,f64,str
2024-11-01,151.72,"""phy"""
2024-11-04,235.53,"""phy"""
2024-11-05,248.56,"""phy"""
2024-11-06,257.29,"""phy"""
2024-11-07,262.31,"""phy"""


# Craft the day-level menus

Day-level menus must satisfy:
- condition (1.2), (2), (6) and (8)
- containing a variety of meals

In [10]:
# During crafting the menus, the condition (6) are already met

meals = dim_meals_raw.filter(
    pl.col('restaurant').is_not_null(),
    pl.col('restaurant').list.contains(restaurant),
    pl.col('schoolyear') == pl.lit(schoolyear)
)


meals_vegan = meals.filter(pl.col("meal_type") == pl.lit("vegan")).select("meal_id")
meals_notvegan = meals.filter(pl.col("meal_type") != pl.lit("vegan")).select("meal_id")

list_meals_vegan = meals_vegan.select('meal_id').to_series().to_list()
list_meals_notvegan = meals_notvegan.select('meal_id').to_series().to_list()

vegan_combo2 = list(combinations(list_meals_vegan, 2))
vegan_combo3 = list(combinations(list_meals_vegan, 3))
vegan_combo4 = list(combinations(list_meals_vegan, 4))

notvegan_combo1 = [(x,) for x in list_meals_notvegan]
notvegan_combo2 = list(combinations(list_meals_notvegan, 2))

In [11]:
def _to_list(list1: list, list2: list):
    return [(*x, *y) for (x, y) in product(list1, list2)]
list_combos = []

# Create combo with 2 vegan and 1 nonvegan
list_combos.extend(_to_list(vegan_combo2, notvegan_combo1))

# Create combo with 3 vegan
list_combos.extend(vegan_combo3)

if restaurant in ["che", "exa", 'vik']:
    # Create combo with 2 vegan and 2 nonvegan
    list_combos.extend(_to_list(vegan_combo2, notvegan_combo2))

    # Create combo with 3 vegan and 1 nonvegan
    list_combos.extend(_to_list(vegan_combo3, notvegan_combo1))

    # Create combo with 4 vegan
    list_combos.extend(vegan_combo4)


In [12]:
is_kela = (
    meals
    .select(
        "meal_id", 
        pl.col('attributes').list.contains('kela').alias("is_kela")
    )
)
is_gluten_free = (
    meals
    .select(
        "meal_id", 
        pl.col('attributes').list.contains('gluten_free').alias("is_gluten_free")
    )
)
meal_type = (
    meals
    .select("meal_id", 'meal_type')
    # .lazy()
)

PREFIXES = ["mon", "tue", "wed", "thu", "fri"]

list_menus_week = []
list_menus_days = []
for prefix in PREFIXES:
    menu_daylevel = (
        pl
        .DataFrame({'meal_id': random.sample(list_combos, min(len(list_combos), NUM_EACH_COMBO))})
        # .lazy()
        .with_row_index()
        .explode('meal_id')
    )

    # Filter out day-level menus not satisfying condition (1.2)
    ids_2kela = (
        menu_daylevel.
        join(is_kela, on='meal_id', how='left')
        # .select(
        #     'index', 'meal_id',
            
        # )
        .group_by('index')
        .agg(pl.col('is_kela').cast(pl.Int64).sum())
        .filter(pl.col('is_kela') >= pl.lit(MIN_KELA_PER_DAY))
        .select('index')
    )
    menu_daylevel = menu_daylevel.join(ids_2kela, on='index', how='inner')


    # Filter out day-level menus not satisfying condition (8)
    ids_gluten_free = (
        menu_daylevel.
        join(is_gluten_free, on='meal_id', how='left')
        # .select(
        #     'index', 'meal_id',
            
        # )
        .group_by('index')
        .agg(pl.col('is_gluten_free').cast(pl.Int64).sum())
        .filter(pl.col('is_gluten_free') >= pl.lit(MIN_GLUTEN_FREE_PER_DAY))
        .select('index')
    )
    menu_daylevel = menu_daylevel.join(ids_gluten_free, on='index', how='inner')


    menu_daylevel = menu_daylevel.select(
        pl.col('index').alias('idx_daylevel'),
        pl.col('meal_id'),
        pl.lit(prefix).alias('weekday')
    )

    list_menus_days.append(menu_daylevel)

    list_menus_week.append(
        menu_daylevel
        .unique('idx_daylevel', keep='first', maintain_order=True)
        .with_row_index()
        .select(
            pl.col('index').alias('idx_weeklevel'),
            'idx_daylevel',
            'weekday'
        )
    )

menus_days = pl.concat(list_menus_days)
menus_week = pl.concat(list_menus_week)

In [14]:
# Filter out week-level menus not having all weekdays
ids_fullweek = (
    menus_week
    .group_by('idx_weeklevel')
    .len()
    .filter(pl.col('len') == 5)
    .select('idx_weeklevel')
)
menus_week = menus_week.join(ids_fullweek, on='idx_weeklevel', how='inner')


menus_week = menus_week.join(menus_days, on=['idx_daylevel', 'weekday'], how='left')


# Filter week-level menus not satisfying condition (3)
ids_max_occur = (
    menus_week
    .group_by(['idx_weeklevel', 'meal_id'])
    .len()
    .filter(pl.col('len') <= MAX_MEAL_OCCURENCES)
    .unique('idx_weeklevel')
    .select('idx_weeklevel')
)
menus_week = menus_week.join(ids_max_occur, on='idx_weeklevel', how='inner')

# Filter out week-level menus not satisfying condition (1.1)
ids_fish = (
    menus_week
    .join(meal_type, on='meal_id', how='left')
    .group_by(['idx_weeklevel', 'meal_type'])
    .len()
    .filter(
        (pl.col('meal_type') == pl.lit('fish'))
        & (pl.col('len') == NUM_FISH_PER_WEEK)
    )
    .unique('idx_weeklevel')
    .select('idx_weeklevel')
)
menus_week = menus_week.join(ids_fish, on='idx_weeklevel', how='inner')


# Limit the max number of week-level menus  
ids_max_week = menus_week.unique('idx_weeklevel').select('idx_weeklevel').head(NUM_DAY_LEVEL_MENUS_2)
menus_week = menus_week.join(ids_max_week, on='idx_weeklevel', how='inner')


menus_week = (
    menus_week
    .select(
        pl.col('idx_weeklevel').alias('index'),
        'weekday',
        'meal_id'
    )
)


menus_week.head()

index,weekday,meal_id
u32,str,i64
548,"""mon""",7564
548,"""mon""",9500050
548,"""mon""",1751
548,"""mon""",1289
554,"""mon""",8999


In [15]:
# (
#     menus_week
#     # .filter(pl.col('index') == 33)
#     # .group_by('weekday')
#     .join(meals.select('meal_id', 'meal_type_1'), on='meal_id', how='left')
#     .filter(pl.col('meal_type_1') == pl.lit('fish'))
#     .group_by('index', 'meal_type_1')
#     .len()
#     .filter(pl.col('len') != 2)
# )

## Add predicted info (biowate, pcs, CO2, pcs_whole)

In [16]:
weekday2date = pl.DataFrame({
    'weekday': PREFIXES,
    'date': pd.date_range(date_start, periods=5)
})

In [17]:
meals_pred_info = (
    menus_week
    .join(weekday2date, on='weekday', how='left')
    .filter(pl.col('meal_id') != pl.lit(-1))
    .select(
        'index',
        'meal_id',
        pl.col('date').dt.date(),
        pl.lit(restaurant).alias('restaurant')
    )
)

meals_pred_info.head()

index,meal_id,date,restaurant
u32,i64,date,str
548,7564,2024-12-02,"""che"""
548,9500050,2024-12-02,"""che"""
548,1751,2024-12-02,"""che"""
548,1289,2024-12-02,"""che"""
554,8999,2024-12-02,"""che"""


In [20]:
# Add predicted POS
pcs_pred = forecast_pcs_per_meal(meals_pred_info)
pcs_pred = pcs_pred.select(
    pl.col('index').cast(pl.UInt32),
    pl.col('date').dt.date(),
    'restaurant', 'meal_id', 'pcs_pred'
    
)

meals_pred_info = meals_pred_info.join(pcs_pred, on=['index', 'date', 'restaurant', 'meal_id'], how='left')



# Add CO2
meals_pred_info = meals_pred_info.join(dim_co2, on='meal_id', how='left')


# Add biowaste
meals_pred_info = meals_pred_info.join(dim_waste, on='meal_id', how='left')



meals_pred_info.head()

index,meal_id,date,restaurant,pcs_pred,co2,waste
u32,i64,date,str,f64,f64,f64
548,7564,2024-12-02,"""che""",114.186166,0.53,0.01
548,9500050,2024-12-02,"""che""",114.877654,0.41,0.109261
548,1751,2024-12-02,"""che""",111.105632,0.57,0.052807
548,1289,2024-12-02,"""che""",118.61,0.54,0.071626
554,8999,2024-12-02,"""che""",107.559729,0.43,0.237928


# Calculate fitness value

In [19]:
THETA_CO2 = 0.5
THETA_WASTE = 0.04
ALPHA_PCS = 2
ALPHA_CO2 = 1
ALPHA_WASTE = 1

In [20]:
menus_fitness = (
    meals_pred_info
    .with_columns(
        (pl.col('pcs_pred') * pl.col('co2')).alias('co2_total_pred'),
        (pl.col('pcs_pred') * pl.col('waste')).alias('waste_total_pred'),
    )
    .group_by(['restaurant', 'date', 'index']).agg(
        pl.col('meal_id').alias('meal_ids'),
        pl.col('pcs_pred').sum().alias('pcs_sum_pred'),
        pl.col('co2_total_pred').sum().alias('co2_sum_pred'),
        pl.col('waste_total_pred').sum().alias('waste_sum_pred')
    )
    .join(dim_pcs_whole, on=['date', 'restaurant'], how='left').rename({'pcs': 'pcs_whole_pred'})
    .with_columns(
        (
            ALPHA_PCS * (pl.col('pcs_sum_pred') / (pl.col('pcs_whole_pred') + 1) - 1).abs()
            + ALPHA_CO2 * (pl.col('co2_sum_pred') / pl.col('pcs_sum_pred')) / THETA_CO2
            + ALPHA_WASTE * (pl.col('waste_sum_pred') / pl.col('pcs_sum_pred')) / THETA_WASTE
        ).alias('fitness')
    )
)

menus_fitness.sort('fitness').head()

restaurant,date,index,meal_ids,pcs_sum_pred,co2_sum_pred,waste_sum_pred,pcs_whole_pred,fitness
str,date,u32,list[i64],f32,f64,f64,f64,f64
"""che""",2024-12-05,404009,"[9050, 9500020, … 8991]",341.551666,60.460151,3.415517,886.54,1.834373
"""che""",2024-12-05,357224,"[6853, 9500150, … 8991]",508.343079,138.48225,9.942953,886.54,1.888316
"""che""",2024-12-03,100936,"[9061, 9060, … 8993]",440.453705,157.462199,4.404537,923.3,2.011946
"""che""",2024-12-03,281973,"[9111, 7583, … 9086]",537.128174,262.254042,5.662167,923.3,2.077806
"""che""",2024-12-02,167166,"[6341, 7595, … 1107]",294.526245,65.818823,2.945262,977.8,2.095136


# To file

In [27]:
(
    menus_fitness
    .filter(pl.col('index') == 404009)
    .explode('meal_ids')
    .join(meals.select('meal_id', 'meal_type'), right_on='meal_id', left_on='meal_ids', how='left')
    .filter(pl.col('meal_type') == pl.lit('fish'))
    .group_by('index', 'meal_type')
    .len()
    # .filter(pl.col('len') != 2)
)

index,meal_type_1,len
u32,str,u32
404009,"""fish""",2
